# Train the clubhead detector (golf-biomechanics-cv)

Trains the YOLOv8n model `backend/app/club.py` expects at `backend/app/models/clubhead.pt`, using the public [golf-club-tracking](https://universe.roboflow.com/club-head-tracking/golf-club-tracking) dataset (~11,479 labeled clubhead images, plain `yolov8` axis-aligned box format — not `yolov8-obb`).

**Before running anything:** `Runtime` → `Change runtime type` → Hardware accelerator → **GPU** (a free T4 is fine). The cell below confirms it's on.

You'll be prompted for your Roboflow API key in a hidden input (not typed into a cell) so it never ends up saved in this notebook's output or committed to git if you push it back to the repo. Get your key from Roboflow → Settings → Roboflow API — and if you've ever pasted a key in plaintext anywhere before, regenerate it first.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q roboflow ultralytics

In [ ]:
from getpass import getpass

api_key = getpass("Roboflow API key: ")

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=api_key)
project = rf.workspace("club-head-tracking").project("golf-club-tracking")
version = project.version(2)
dataset = version.download("yolov8")  # NOT yolov8-obb -- app/club.py reads axis-aligned box.xyxy
print(dataset.location)

## Train

50 epochs at 640px, matching `train_clubhead.py`'s defaults. In practice each epoch over ~11.5k images tends to run more like 3–4 minutes on a free T4 (not the 30–60 min total this notebook originally guessed) — so a full 50-epoch run is more like 3+ hours. Two changes below to make that more manageable:

- `batch=-1` turns on Ultralytics' AutoBatch, which picks the largest batch size that fits the T4's memory instead of the small default (16) — batching more images per step is the single biggest lever for wall-clock speed on a GPU that isn't otherwise saturated by a model this small.
- `patience=15` stops training early if validation performance hasn't improved in 15 epochs. `best.pt` (what actually gets used) is checkpointed continuously regardless of when training stops, so this only cuts wasted time, not quality — clubhead detection is a small, single-class task that typically plateaus well before epoch 50.

**Free-tier disconnect risk:** Colab can disconnect an idle tab, and a 1–3 hour run risks that if you walk away. Check in occasionally, or if you do get disconnected, rerun the dataset-download cell then resume from the last checkpoint instead of starting over:
```python
model = YOLO("runs/detect/train/weights/last.pt")
results = model.train(resume=True)
```

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(data=f"{dataset.location}/data.yaml", epochs=50, imgsz=640, batch=-1, patience=15)

## Download the trained weights

This downloads `best.pt` to your computer via the browser. Put it at `backend/app/models/clubhead.pt` in the repo (create the `models/` folder if it doesn't exist) — `app/club.py` lazy-loads it from exactly that path, no other code changes needed.

In [ ]:
from pathlib import Path
from google.colab import files

best_weights = Path(results.save_dir) / "weights" / "best.pt"
print(f"Best weights: {best_weights}")
files.download(str(best_weights))